In [1]:
import pandas as pd

paper = pd.read_csv("/content/drive/MyDrive/Project-Paper-RCM-System-Seminar/data/new_data/paper_full.csv")
paper_keyword = pd.read_csv("/content/drive/MyDrive/Project-Paper-RCM-System-Seminar/data/new_data/paper_keyword.csv")
venue = pd.read_csv("/content/drive/MyDrive/Project-Paper-RCM-System-Seminar/data/new_data/venue.csv").rename(columns={"name": "target_venue"})
keyword = pd.read_csv("/content/drive/MyDrive/Project-Paper-RCM-System-Seminar/data/new_data/keyword.csv").rename(columns={"name": "keyword"})

In [2]:
paper.drop("embedding", axis=1, inplace=True)

### **Preprocessing Data**

##### 1. Function to join and aggregate

In [3]:
def join_and_aggregate(paper, venue, paper_keyword, keyword):

  # Join table
  df = (
    paper
    .merge(venue, left_on="venue_id", right_on="id", how="inner")
    .merge(paper_keyword, left_on="doi", right_on="paper_doi", how="inner")
    .merge(keyword, left_on="keyword_id", right_on="id", how="inner")
  )

  # Choose necessary cols
  df = df[["doi", "venue_id", "title", "abstract", "keyword", "target_venue"]]

  # Combine keyword into list per doi
  df = (
    df.groupby(["doi", "venue_id", "title", "abstract", "target_venue"])
    ["keyword"]
    .apply(list)
    .reset_index()
  )

  return df

In [4]:
df = join_and_aggregate(paper, venue, paper_keyword, keyword)

In [ ]:
df.head()

,doi,venue_id,title,abstract,target_venue,keyword
0,10.1001/dmp.2012.4,2747.0,Core Competencies for Disaster Medicine and Pu...,"ABSTRACTEffective preparedness, response, and ...",Disaster Medicine and Public Health Preparedness,"[Preparedness, Core competency, Workforce, Pub..."
1,10.1001/jama.2022.21243,960.0,Guidelines for Reporting Outcomes in Trial Pro...,ImportanceComplete information in a trial prot...,JAMA,"[Medicine, Protocol (science), Consolidated St..."
2,10.1001/jama.2023.1044,960.0,Appropriateness of Cardiovascular Disease Prev...,This study examines the appropriateness of art...,JAMA,"[Medicine, Disease, MEDLINE, Artificial intell..."
3,10.1001/jama.2023.14217,960.0,Creation and Adoption of Large Language Models...,ImportanceThere is increased interest in and p...,JAMA,"[Medicine, Provisioning, Agency (philosophy), ..."
4,10.1001/jama.2023.4221,960.0,Emulation of Randomized Clinical Trials With N...,ImportanceNonrandomized studies using insuranc...,JAMA,"[Medicine, Emulation, Randomized controlled tr..."


##### 2. Processing text

In [5]:
import re
import ast
import pandas as pd

class DataPreprocessor:
  """
  A utility class for preprocessing text and keyword fields in a DataFrame.

  This class provides methods to:
  - Clean and normalize text (title, abstract).
  - Convert string representations of lists.
  - Clean and filter keyword lists.
  - Apply all transformations to a DataFrame in one step.
  """
  def __init__(self) -> None:
    pass

  # ----------------------------------------------------------------------------
  @staticmethod
  def preprocess_text(text: str) -> str:
    """
    Normalize and clean text for 'title' and 'abstract' fields.

    Args:
      text (str): Input text string.

    Returns:
      str: Cleaned and normalized text.
           Returns an empty string if input is not a string.
    """
    if not isinstance(text, str):
      return

    text = text.lower()

    # Remove complex math formulas/LaTeX sections
    text = re.sub(r"\$\$.*?\$\$", "", text)
    text = re.sub(r"\$.*?\$", "", text)
    text = re.sub(r"\\\(.*?\\\)", "", text)
    text = re.sub(r"\\\[.*?\\\]", "", text)
    text = re.sub(r"\\begin.*?\\end", "", text)

    # Remove other brackets
    text = re.sub(r"\([^)]*\)", "", text)
    text = re.sub(r"\[.*?\]", "", text)
    text = re.sub(r"{.*?}", "", text)

    # Other removals
    text = re.sub(r"http[^ ]*", "", text)
    text = re.sub(r"[^0-9a-zA-ZÀ-ỹ\s]", "", text)
    text = re.sub(r'\s+', " ", text)

    return text.strip()

  # ----------------------------------------------------------------------------
  @staticmethod
  def clean_keyword(keywords: list[str]) -> list[str]:
    """
    Clean and normalize keyword list.

    - Remove bracket content
    - Remove redundant acronyms if full-form exists
    - Lowercase
    """
    if not keywords:
      return []

    # Normalize & remove bracket content
    cleaned = [
      re.sub(r"\([^)]*\)", "", kw).strip()
      for kw in keywords
      if isinstance(kw, str) and kw.strip()
    ]

    # Extract full forms (>= 2 words)
    full_forms = [kw for kw in cleaned if len(kw.split()) > 1]

    # Generate acronyms from full forms
    derived_acronyms = {
      "".join(w[0] for w in ff.split() if w[0].isalpha()).upper()
      for ff in full_forms
    }

    def is_abbreviation(word: str) -> bool:
      letters = [ch for ch in word if ch.isalpha()]
      if not letters or len(letters) > 5:
        return False
      upper_ratio = sum(ch.isupper() for ch in letters) / len(letters)
      return upper_ratio >= 0.6

    result = []
    for kw in cleaned:
      if is_abbreviation(kw) and kw.upper() in derived_acronyms:
        continue
      result.append(kw.lower())

    return result

  # ----------------------------------------------------------------------------
  def transform(self, df: pd.DataFrame) -> pd.DataFrame:
    """
    Apply preprocessing pipeline to a DataFrame.

    Args:
      df (pd.DataFrame): Input DataFrame with columns 'title', 'abstract', and 'keyword'.

    Returns:
      pd.DataFrame: A new DataFrame with cleaned and normalized values.
    """
    df = df.copy()

    if "title" in df.columns:
      df['title'] = df['title'].apply(self.preprocess_text)

    if "abstract" in df.columns:
      df['abstract'] = df['abstract'].apply(self.preprocess_text)

    if "keyword" in df.columns:
      df["keyword"] = df["keyword"].apply(self.clean_keyword)

    return df

In [6]:
processed_df = DataPreprocessor().transform(df)

In [7]:
processed_df['keyword'][30000]

['software engineering', 'code smells', 'refactoring']

##### Processing keyword

In [8]:
def normalize_kw(kw: str) -> str:
  kw = kw.lower().strip()
  kw = re.sub(r"[^a-z0-9\s\-]", "", kw)
  kw = re.sub(r"\s+", " ", kw)
  return kw

processed_df["keyword"] = processed_df["keyword"].apply(
  lambda kws: " ".join(normalize_kw(kw) for kw in kws)
  if isinstance(kws, list) else ""
)

In [9]:
processed_df['keyword'][30000]

'software engineering code smells refactoring'

### **Feature Engineering**

##### 1. Applying Chi-square for 'abstract' field to filter important tokens

**Công thức tính Chi-square:**

\begin{equation}
\chi^2 = \frac{N(AD - BC)^2}{(A + B)(C + D)(A + C)(B + D)}
\end{equation}

Trong đó:<br>
- $N$ là tổng số lượng bài báo (thường là $N = A + B + C + D$).
- $A$ là số bài báo trong journal $j$ / conference $c$ có chứa token.
- $B$ là số bài báo không thuộc journal $j$ / conference $c$  có chứa token.
- $C$ là số bài báo trong journal $j$ / conference $c$ không chứa token.
- $D$ là số bài báo không thuộc journal $j$ / conference $c$ không chứa token.

Nếu $\chi^2$ cao → token có liên hệ mạnh với venue đó → nên giữ lại.<br>
Nếu $\chi^2$ thấp → token không mang tính phân biệt → nên loại bỏ.

In [ ]:
# import numpy as np
# from tqdm import tqdm
# from collections import defaultdict
# from sklearn.feature_extraction.text import CountVectorizer

# def calculate_chi_square(
#   dataset,
#   text_field="abstract",
#   label_field="venue_id",
#   min_df=5,
#   max_df=0.95
#   top_k_per_class=50
# ):
#   texts = dataset[text_field].tolist()
#   labels = dataset[label_field].tolist()

#   vectorizer = CountVectorizer(
#     binary=True,
#     lowercase=True,
#     min_df=min_df,
#     max_df=max_df
#   )

#   X = vectorizer.fit_transform(texts)   # CSR sparse
#   vocab = vectorizer.get_feature_names_out()
#   N, V = X.shape

#   # venue → doc indices
#   label2idx = defaultdict(list)
#   for i, y in enumerate(labels):
#     label2idx[y].append(i)

#   # token document frequency
#   token_df = X.sum(axis=0).A1  # C + A

#   selected_tokens = set()

#   for label, idx_in in tqdm(label2idx.items(), desc="Chi-square per venue"):
#     idx_in = np.array(idx_in)
#     n_in = len(idx_in)
#     n_out = N - n_in

#     X_in = X[idx_in]
#     A = X_in.sum(axis=0).A1
#     C = token_df - A

#     B = n_in - A
#     D = n_out - C

#     numerator = (A * D - B * C) ** 2 * N
#     denominator = (A + B) * (A + C) * (B + D) * (C + D)

#     chi = np.divide(
#       numerator,
#       denominator,
#       out=np.zeros_like(numerator, dtype=float),
#       where=denominator != 0
#     )

#     top_idx = np.argsort(chi)[-top_k_per_class:]
#     for i in top_idx:
#       selected_tokens.add(vocab[i])

#   return list(selected_tokens)

In [ ]:
# abstract_chi2 = calculate_chi_square(processed_df)

Chi-square per venue: 100%|██████████| 11753/11753 [01:08<00:00, 171.25it/s]


In [ ]:
# # Store important vocabs
# import pickle

# with open("/content/drive/MyDrive/Project-Paper-RCM-System-Seminar/data/new_data/chi_square_vocab.pkl", "wb") as f:
#   pickle.dump(abstract_chi2, f)

# Load abstract tokens
import pickle

with open("/content/drive/MyDrive/Project-Paper-RCM-System-Seminar/data/new_data/chi_square_vocab.pkl", "rb") as f:
  abstract_tokens = pickle.load(f)

##### 2. Apply TF-IDF for title and filtered abstract vocabs

In [ ]:
import pandas as pd
from scipy.sparse import csr_matrix
from typing import Optional, Tuple, cast
from sklearn.feature_extraction.text import TfidfVectorizer

def build_tfidf_embedding(
  texts: pd.Series,
  vocabulary: Optional[list[str]] = None,
  ngram_range: tuple[int, int] = (1, 2),
  min_df: int = 5,
  max_df: float = 0.95,
) -> Tuple[TfidfVectorizer, csr_matrix]:
  """
  Build TF-IDF embeddings for a text field.

  Args:
    texts (pd.Series): Text data (title or abstract).
    vocabulary (list[str], optional): Predefined vocabulary (used for abstract after Chi-square).
    ngram_range (tuple): N-gram range for TF-IDF.
    min_df (int): Min document frequency.
    max_df (float): Max document frequency.

  Returns:
    vectorizer (TfidfVectorizer): Fitted TF-IDF vectorizer.
    embeddings (csr_matrix): Sparse TF-IDF matrix.
  """

  vectorizer = TfidfVectorizer(
    lowercase=True,
    vocabulary=vocabulary,
    ngram_range=ngram_range,
    min_df=min_df if vocabulary is None else 1,
    max_df=max_df if vocabulary is None else 1.0
  )

  embeddings = cast(csr_matrix ,vectorizer.fit_transform(texts))

  return vectorizer, embeddings

In [ ]:
tfidf_title, W_t = build_tfidf_embedding(
  processed_df['title'],
  ngram_range=(1, 2),
  min_df=5,
  max_df=0.95
)

In [ ]:
tfidf_abstract, W_a = build_tfidf_embedding(
  processed_df['abstract'],
  vocabulary=abstract_tokens,
)

In [ ]:
tfidf_keyword, W_k = build_tfidf_embedding(
  processed_df['keyword'],
  ngram_range=(1, 1),
  min_df=3,
  max_df=0.9
)

In [ ]:
print(f"Total vocabularies of title field: {len(tfidf_title.get_feature_names_out())}")
print(f"Total vocabularies of title field: {len(tfidf_abstract.get_feature_names_out())}")
print(f"Total vocabularies of title field: {len(tfidf_keyword.get_feature_names_out())}")

Total vocabularies of title field: 74197
Total vocabularies of title field: 65541
Total vocabularies of title field: 20831


In [ ]:
W_t.shape, W_a.shape, W_k.shape

((223879, 74197), (223879, 65541), (223879, 20831))

##### 3. Apply weighted and combine vector

In [ ]:
import pickle
import scipy.sparse as sp
from sklearn.metrics.pairwise import cosine_similarity

# 1. Define weight
# The title is the most important, followed by the keyword, and the abstract is less important due to a lot of noise.
WEIGHT_TITLE = 3.0
WEIGHT_KEYWORD = 2.0
WEIGHT_ABSTRACT = 1.0

# 2. Nhân trọng số trực tiếp vào ma trận
W_t_weighted = W_t * WEIGHT_TITLE
W_a_weighted = W_a * WEIGHT_ABSTRACT
W_k_weighted = W_k * WEIGHT_KEYWORD

# 3. Combine 3 matrix to 1 (Horizontal Stack)
final_paper_vectors = sp.hstack([W_t_weighted, W_a_weighted, W_k_weighted], format='csr')
print(f"Shape of final database matrix: {final_paper_vectors.shape}")

Shape of final database matrix: (223879, 160569)


In [ ]:
with open("/content/drive/MyDrive/Project-Paper-RCM-System-Seminar/data/new_data/tfidf_models.pkl", "wb") as f:
  pickle.dump({
    "vec_title": tfidf_title,
    "vec_abstract": tfidf_abstract,
    "vec_keyword": tfidf_keyword
  }, f)

In [ ]:
sp.save_npz("/content/drive/MyDrive/Project-Paper-RCM-System-Seminar/data/new_data/database_tfidf_matrix.npz", final_paper_vectors)

### **Recommendation Workflow**

##### 1. Process user input

In [10]:
import re
from typing import List, Union, Dict, Optional

class UserInputProcessor:
  def __init__(self):
    pass

  # ----------------------------------------------------------------------------
  def preprocess_text(self, text: str) -> str:
    """
    Normalize and clean text for 'title' and 'abstract' fields.

    Args:
      text (str): Input text string.

    Returns:
      str: Cleaned and normalized text.
           Returns an empty string if input is not a string.
    """
    if not isinstance(text, str):
      return ""

    text = text.lower()

    # Remove complex math formulas/LaTeX sections
    text = re.sub(r"\$\$.*?\$\$", "", text)
    text = re.sub(r"\$.*?\$", "", text)
    text = re.sub(r"\\\(.*?\\\)", "", text)
    text = re.sub(r"\\\[.*?\\\]", "", text)
    text = re.sub(r"\\begin.*?\\end", "", text)

    # Remove other brackets
    text = re.sub(r"\([^)]*\)", "", text)
    text = re.sub(r"\[.*?\]", "", text)
    text = re.sub(r"{.*?}", "", text)

    # Other removals
    text = re.sub(r"http[^ ]*", "", text)
    text = re.sub(r"[^0-9a-zA-ZÀ-ỹ\s]", "", text)
    text = re.sub(r'\s+', " ", text)

    return text.strip()

  # ----------------------------------------------------------------------------
  def normalize_keywords(self, keyword: Union[str, List[str], None]) -> str:
    """
    Convert keywords input (string, list, or None) into a single normalized string.

    Args:
      keyword: Can be:
        - None
        - String (comma-separated or space-separated)
        - List of strings

    Returns:
      str: Space-separated lowercase keywords

    Examples:
      normalize_keywords(['Deep Learning', 'NLP']) -> 'deep learning nlp'
      normalize_keywords('Deep Learning, NLP') -> 'deep learning nlp'
      normalize_keywords(None) -> ''
    """
    if not keyword:
      return ""

    # If it's a list, join with spaces
    if isinstance(keyword, list):
      keyword = " ".join(keyword)

    # If it's a string, clean it
    if isinstance(keyword, str):
      # Remove common separators and normalize
      keyword = re.sub(r'[,;|]+', ' ', keyword)
      keyword = re.sub(r'\s+', ' ', keyword)
      return keyword.lower().strip()

    return ""

  # ----------------------------------------------------------------------------
  def process_user_input(
    self,
    title: Optional[str] = None,
    abstract: Optional[str] = None,
    keyword: Union[str, List[str], None] = None
  ) -> Dict[str, str]:
    """
    Preprocess user input fields.

    Args:
      title: Paper title
      abstract: Paper abstract
      keyword: None/ String (comma-separated or space-separated)/ List of strings

    Returns:
      dict with keys: title, abstract, keyword
    """
    processed_title = self.preprocess_text(title)
    processed_abstract = self.preprocess_text(abstract)
    processed_keyword = self.normalize_keywords(keyword)

    return {
      "title": processed_title,
      "abstract": processed_abstract,
      "keyword": processed_abstract
    }

##### 2. Load weighted models

In [11]:
import pickle
import scipy.sparse as sp
from typing import Optional, Tuple, cast
from sklearn.feature_extraction.text import TfidfVectorizer

def load_resources(
  model_path: str = "resources/tfidf_models.pkl",
  matrix_path: str = "resources/database_matrix.npz"
) -> Tuple[Dict[str, TfidfVectorizer], sp.csr_matrix]:
  """
  Load resources lên RAM.
  Returns:
    models (dict): {"title": vec, ...}
    matrix (csr_matrix): Database vectors
  """
  try:
    # 1. Load Models
    with open(model_path, "rb") as f:
      models = pickle.load(f)

    # 2. Load Matrix
    matrix = sp.load_npz(matrix_path)

    print("Resources loaded successfully!")
    return models, matrix

  except FileNotFoundError as e:
    print(f"Error loading resources: {e}")
    return {}, None

In [12]:
loaded_models, loaded_matrix = load_resources(
  model_path="/content/drive/MyDrive/Project-Paper-RCM-System-Seminar/data/new_data/tfidf_models.pkl",
  matrix_path="/content/drive/MyDrive/Project-Paper-RCM-System-Seminar/data/new_data/database_tfidf_matrix.npz"
)

Resources loaded successfully!


##### 3. Recommnedation Engine

In [13]:
from sklearn.metrics.pairwise import cosine_similarity

def content_based_recommendation(
  user_title: str,
  user_abstract: str,
  user_keyword: str,
  models: Dict[str, TfidfVectorizer],
  database_matrix: sp.csr_matrix,
  df_papers: pd.DataFrame,
  weight_title: float = 3.0,
  weight_keyword: float = 2.0,
  weight_abstract: float = 1.0,
  top_k: int = 10
  ):

  """
  Generate content-based recommendations.
  """
  processor = UserInputProcessor()

  # 1. Preprocess User Input
  inputs = processor.process_user_input(
    title=user_title,
    abstract=user_abstract,
    keyword=user_keyword
  )

  # 2. Transform (Use .transform, NOT .fit_transform)
  try:
    user_W_t = models['vec_title'].transform([inputs["title"]])
    user_W_a = models['vec_abstract'].transform([inputs["abstract"]])
    user_W_k = models['vec_keyword'].transform([inputs["keyword"]])
  except KeyError as e:
    raise ValueError(f"Model dictionary is missing key: {e}")

  # 3. Apply Weights
  user_W_t = user_W_t * weight_title
  user_W_a = user_W_a * weight_abstract
  user_W_k = user_W_k * weight_keyword

  # 4. Stack Vectors
  user_final_vector = sp.hstack([user_W_t, user_W_a, user_W_k], format='csr')

  # 5. Cosine Similarity
  similarity_scores = cosine_similarity(user_final_vector, database_matrix)

  # 6. Retrieve Top K
  top_indices = similarity_scores[0].argsort()[::-1][:top_k]
  top_scores = similarity_scores[0][top_indices]

  # 7. Return Result
  results = df_papers.iloc[top_indices].copy()
  results['similarity_score'] = top_scores

  return results

In [ ]:
user_title = "SimCPSR: Simple Contrastive Learning for Paper Submission Recommendation System"
user_abstract = """The recommendation system plays a vital role in many areas, especially academic fields, to support researchers in submitting and
increasing the acceptance of their work through the conference or journal
selection process. This study proposes a transformer-based model using
transfer learning as an efficient approach for the paper submission recommendation system. By combining essential information (such as the
title, the abstract, and the list of keywords) with the aims & scopes
of journals, the model can recommend the Top K journals that maximize the acceptance of the paper. Our model had developed through
two states: (i) Fine-tuning the pre-trained language model (LM) with
a simple contrastive learning framework. We utilized a simple supervised contrastive objective to fine-tune all parameters, encouraging the
LM to learn the document representation effectively. (ii) The fine-tuned
LM was then trained on different combinations of the features for the
downstream task. This study suggests a more advanced method for enhancing the efficiency of the paper submission recommendation system
compared to previous approaches when we respectively achieve 0.5173,
0.8097, 0.8862, 0.9496 for Top 1, 3, 5, 10 accuracies on the test set for
combining the title, abstract, and keywords as input features. Incorporating the journals’ aims and scopes, our model shows an exciting result
by getting 0.5194, 0.8112, 0.8866, 0.9496 respective to Top 1, 3, 5, and
10."""
user_keyword = "paper submission recommendation, contrastive learning, sentence embedding, recommendation system"

In [15]:
result = content_based_recommendation(
  user_title,
  user_abstract,
  user_keyword,
  models=loaded_models,
  database_matrix=loaded_matrix,
  df_papers=processed_df
)

In [16]:
result[['venue_id', 'title', 'keyword', 'target_venue', 'similarity_score']]

,venue_id,title,keyword,target_venue,similarity_score
112145,202.0,knowledgeflow contrastive learning for recomme...,knowledge graph contrastive learning feature f...,Information Fusion,0.354220
105953,195.0,sorcl socialreachabilitydriven contrastive lea...,friend recommendation graph neural networks so...,Expert Systems with Applications,0.349129
88739,1313.0,classificationwise and clusterwise contrastive...,contrastive learning graph neural network reco...,Applied Soft Computing,0.323794
60249,255.0,diagnosis recommendation system,medical diagnosis recommendation systems data ...,Proceedings of the Future Technologies Conference,0.306404
107210,195.0,rsgcl randomized svdbased graphenhanced contra...,recommendation system contrastive learning gra...,Expert Systems with Applications,0.305276
29808,10108.0,simple flowbased contrastive learning for bert...,flow model contrastive learning deep learning,International Conference on Sensing and Imaging,0.301157
119762,1203.0,hierarchical neighborenhanced graph contrastiv...,recommender systems contrastive learning graph...,Knowledge Based Systems,0.295073
76390,5418.0,suggestosphere a sports recommendation system,sports recommendation system machine learning ...,International Conference on Information and Co...,0.280657
124226,856.0,grade generative graph contrastive learning fo...,multimodal recommendation systems graph convol...,Neurocomputing,0.276977
106165,195.0,glscl graph local similarity contrastive learn...,recommender system contrastive learning graph ...,Expert Systems with Applications,0.276750


In [17]:
result['target_venue'][76390]

'International Conference on Information and Communication Technology for Intelligent Systems'

### **Evaluation**

##### Rule-based Filtering - high confidence

In [ ]:
# def label_venue(venue_name):
#   if not venue_name:
#     return 'unknown'

#   name = venue_name.lower().strip()

#   # 1. Xử lý các Journal ngoại lệ nổi tiếng trước
#   journal_whitelist = [
#     'nature', 'science', 'plos one', 'ieee access', 'heliyon',
#     'proceedings of the ieee', 'proceedings of the national academy of sciences',
#     'f1000research', 'pnas', 'royal society open science', 'database (oxford)',
#     'the photogrammetric record',
#   ]
#   if name in journal_whitelist:
#     return 'journal'

#   # 2. Kiểm tra keywords CONFERENCE (High priority)
#   conference_keywords = [
#     'conference', 'symposium', 'workshop', 'congress', 'meeting',
#     'proc.', 'proceedings', 'symp.', 'conf.', 'convention', 'iclr',
#     'cvpr', 'icml', 'neurips', 'nips', 'acl', 'emnlp', 'ijcai', 'aaai',
#     'acm', 'volume', 'cirp', 'procedia', 'ifac', 'sae', 'forum', 'international',
#     'iot and big data technologies for health care', 'Congreso Nacional de Ingeniería Biomédica',
#     'oxford', 'colloquium', 'seminar', 'retreat', 'school', 'summer school', 'winter school',
#     'hackathon', 'challenge', 'competition', 'companion', 'extended abstracts', 'abstracts volume',
#     'annual meeting', 'technical meeting', 'user meeting', 'world cup', 'cup', 'special session',
#     'invited session', 'ACM International Conference Proceedings Series', 'sae technical paper series',
#     'archives of the photogrammetry', 'international archives of the photogrammetry',
#     'naacl', 'coling', 'ecai', 'iccv', 'eccv', 'bmvc', 'kdd', 'sigir', 'cikm', 'wsdm', 'www', 'thewebconf',
#     'icse', 'fse', 'esec', 'ase', 'issta', 'saner', 're', 'infocom', 'icc', 'globecom', 'mobicom',
#     'MobiSys', 'SenSys', 'ipsn', 'siggraph', 'siggraph asia', 'chi', 'uist', 'cscw', 'vis', 'eurovis',
#     'miccai', 'isbi', 'ipmi', 'eccai', 'ifac papersonLine', 'ifip', 'remote sensing and spatial information sciences',
#     'robot world cup', 'robocup', 'lecture notes in', 'lncs', 'lnai', 'lnbi', 'lnicst', 'lnbip', 'ccis',
#     'ceur workshop proceedings', 'acm icps', 'aip conference proceedings', 'spie proceedings', 'ifac papersonline', 'ifip',
#     'miccai challenge', 'grand challenge', 'forum in', 'forum of', 'international summit', 'global summit', 'conclave',
#     'school on', 'course on'
#   ]
#   # LNCS của Springer thường chứa bài conference
#   if 'lecture notes in' in name:
#     return 'conference'

#   for kw in conference_keywords:
#     if kw in name:
#       # Loại trừ trường hợp PNAS nếu logic trên chưa bắt
#       if kw == 'proceedings' and 'national academy of sciences' in name:
#         continue
#       return 'conference'

#   # 3. Kiểm tra keywords JOURNALS
#   journal_keywords = [
#     'journal', 'transactions', 'review', 'letters', 'annals',
#     'bulletin', 'quarterly', 'magazine', 'communications', 'acta',
#     'current opinion', 'trends in', 'reviews', 'reports', 'monthly', 'bimonthly',
#     'archives', 'communication', 'notes', 'acs', 'elsevier', 'wiley', 'rsc', 'ieee',
#     'iop', 'spinger', 'annual review', 'open research', 'npj', 'bmj', 'jama', 'plos',
#     'elife', 'peerj', 'wellcome open research', 'materials today', 'cell reports',
#     'cell systems', 'advanced materials', 'advanced science', 'phys. chem. chem. phys.',
#     'j. mater. chem.','chem. soc. rev.','crystengcomm','soft matter','rsc adv'
#   ]
#   for kw in journal_keywords:
#     if kw in name:
#       return 'journal'

#   return 'unknown'

# # Create venue_type column based on target_venue column
# processed_df['venue_type'] = processed_df['target_venue'].apply(label_venue)

In [ ]:
processed_df.drop('venue_type', axis=1, inplace=True)

In [ ]:
# processed_df['venue_type'].value_counts()

,count
venue_type,
conference,126556
unknown,56220
journal,41103


In [ ]:
# unknown_venues = processed_df[processed_df['venue_type'] == 'unknown']

In [ ]:
# unknown_venues = (
#   processed_df[processed_df['venue_type'] == 'unknown']
#   .groupby('target_venue')
#   .size()
#   .reset_index(name='count')
#   .sort_values('count', ascending=False)
# )

##### 1. Proportional Stratified Sampling

In [18]:
# 1. Filter venues having less papers (ví dụ < 5 bài)
venue_counts = processed_df['venue_id'].value_counts()
valid_venues = venue_counts[venue_counts >= 5].index
filtered_df = processed_df[processed_df['venue_id'].isin(valid_venues)]

In [19]:
print(f"Original papers: {len(processed_df)}")
print(f"Papers in valid venues (>=5 papers): {len(filtered_df)}")
print(f"Dropped {len(processed_df) - len(filtered_df)} papers from sparse venues.")

Original papers: 223879
Papers in valid venues (>=5 papers): 212251
Dropped 11628 papers from sparse venues.


In [20]:
# 2. Take 10% dataset
from sklearn.model_selection import train_test_split

_, test_set = train_test_split(
  filtered_df,
  test_size=0.1,
  random_state=42,
  stratify=filtered_df['venue_id']
)

print(f"Test set size: {len(test_set)}")
print(f"Number of unique venues in test set: {test_set['venue_id'].nunique()}")

Test set size: 21226
Number of unique venues in test set: 4838


In [21]:
test_set.head(5)

,doi,venue_id,title,abstract,target_venue,keyword
113538,10.1016/j.ipm.2025.104153,194.0,realexp decoupling correlation bias in shapley...,deep learning has achieved significant success...,Information Processing and Management,deep learning explained artificial intelligenc...
143782,10.1038/s41524-021-00520-w,3105.0,twostep machine learning enables optimized nan...,abstract in materials science the discovery of...,npj Computational Materials,bayesian optimization computer science absorba...
22684,10.1007/978-3-030-78086-9_30,10931.0,software integrity and validation using crypto...,a significant aspect of software integrity is ...,International Symposium on Cyber Security Cryp...,integrity assurance composability cryptography...
49054,10.1007/978-3-031-75164-6_14,2774.0,hate speech detection using glove and bert,the increasing prevalence of hate speech on pl...,International Conference on Artificial Intelli...,hate speech naive bayes machine learning decis...
183306,10.1145/2010324.1964981,6867.0,make it home,we present a system that automatically synthes...,ACM Transactions on Graphics,simulated annealing computer science visibilit...


In [23]:
test_set.to_csv("/content/drive/MyDrive/Project-Paper-RCM-System-Seminar/data/new_data/test_benchmark_data.csv", index=False)

2. Benchmark for Content-Based Filtering (CBF) with Hit Rate@k

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import scipy.sparse as sp
from typing import Dict, List
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

def run_cbf_benchmark(
  test_set: pd.DataFrame,
  full_df: pd.DataFrame,
  models: Dict[str, TfidfVectorizer],
  database_matrix: sp.csr_matrix,
  weight_title: float = 3.0,
  weight_keyword: float = 2.0,
  weight_abstract: float = 1.0
):
  """
  Run benchmarking evaluate Hit Rate @ K (Strict Match) for Content-Based Filtering.
  """

  # 1. Initialize counter variable
  k_metrics = [1, 3, 5, 10]
  hits = {k: 0 for k in k_metrics}

  total_samples = len(test_set)
  processor = UserInputProcessor()

  print(f"Starting Benchmark on {total_samples} papers...")
  print(f"Dataset Shape: {full_df.shape} | Matrix Shape: {database_matrix.shape}")

  # 2. Loop through each papers in Test set
  for idx, row in tqdm(test_set.iterrows(), total=total_samples, desc="Benchmarking"):

    # Get information for Ground Truth
    true_doi = row.get('doi')
    true_venue_id = row.get('venue_id')

    if pd.isna(true_venue_id):
      total_samples -= 1
      continue

    # Simulate User Input Stream (Preprocess & Vectorize)
    inputs = processor.process_user_input(
      title=str(row.get('title', '')),
      abstract=str(row.get('abstract', '')),
      keyword=str(row.get('keyword', ''))
    )

    try:
      user_W_t = models["vec_title"].transform([inputs["title"]]) * weight_title
      user_W_a = models["vec_abstract"].transform([inputs["abstract"]]) * weight_abstract
      user_W_k = models["vec_keyword"].transform([inputs["keyword"]]) * weight_keyword

      # Combine vectors (Horizontal Stack)
      user_final_vector = sp.hstack([user_W_t, user_W_a, user_W_k], format='csr')
    except Exception as e:
      continue

    # Calculate Similarity & Ranking
    similarity_scores = cosine_similarity(user_final_vector, database_matrix).flatten()

    # Get top 20 indices having highest score
    top_indices = similarity_scores.argsort()[::-1][:20]

    # Filter & Strict Match Check
    candidate_papers = full_df.iloc[top_indices]

    # Remove the article being tested (Self-exclusion)
    filtered_recs = candidate_papers[candidate_papers['doi'] != true_doi]

    # Get the Venue ID list of the suggested items
    rec_venue_ids = filtered_recs['venue_id'].tolist()

    # Check Hit at K levels
    for k in k_metrics:
      top_k_venues = rec_venue_ids[:k]

      if true_venue_id in top_k_venues:
        hits[k] += 1

  # 3. Export report
  print("\n" + "="*50)
  print(f"CONTENT-BASED FILTERING PERFORMANCE REPORT")
  print(f"Tested Samples: {total_samples}")
  print("="*50)

  results = {}
  for k in k_metrics:
    acc = (hits[k] / total_samples) * 100
    results[f"Hit@{k}"] = acc
    print(f"Hit Rate @ {k:<2}: {acc:.2f}%  ({hits[k]}/{total_samples})")

  print("="*50)
  return results

In [ ]:
benchmark_results = run_cbf_benchmark(
  test_set=test_set,
  full_df=processed_df,
  models=loaded_models,
  database_matrix=loaded_matrix
)

Starting Benchmark on 21226 papers...
Dataset Shape: (223879, 6) | Matrix Shape: (223879, 160569)


Benchmarking: 100%|██████████| 21226/21226 [5:43:07<00:00,  1.03it/s]


CONTENT-BASED FILTERING PERFORMANCE REPORT
Tested Samples: 21226
Hit Rate @ 1 : 7.39%  (1568/21226)
Hit Rate @ 3 : 14.28%  (3032/21226)
Hit Rate @ 5 : 18.40%  (3906/21226)
Hit Rate @ 10: 25.32%  (5375/21226)
